In [1]:
import numpy as np

import sys
sys.path.insert(0, "../src")
import numpy as np
from energy import load_framework, load_force_field, pairwise_lj_params, energy_fast  

def widom(N, T, positions, eps_pair, sigma_pair, L, rng, n_blocks=10):
    random_positions = rng.random((N,3))*L
    U = np.array([energy_fast(pos, positions, eps_pair, sigma_pair, L) for pos in random_positions])
    boltzmann_factors = boltzmann(U, T)
    boltzmann_mean_value = np.mean(boltzmann_factors)
    boltzman_reshape = boltzmann_factors.reshape(n_blocks, -1).mean(axis=1)
    #standard error of the mean
    sem = boltzman_reshape.std() / np.sqrt(n_blocks)
    return boltzmann_mean_value, sem

def boltzmann(U, T):
    return np.exp(-U / T)

def henry_coef(boltzmann_mean, T, rho_fw, R=8.31446261815324):
    return boltzmann_mean / (rho_fw * T*R)

def rho_framework(cif_path, avogadro=6.02214076*10**(23) ):
    positions, symbols, L, masses, atoms = load_framework(cif_path)
    total_mass = np.sum(masses)/1000/avogadro 
    V_framework = (L*10**(-10))**3 
    return total_mass/V_framework




In [15]:
N = 10000
T = 298 #K
cif_path='/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/data/IRMOF-1.cif'
positions, symbols, L, masses, atoms = load_framework(cif_path)
ff_path = "/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/data/force_field/generic_dreiding_uff_mixing_rules.def"
lj_params = load_force_field(ff_path, elements=("O_", "C_", "H_", "Zn_"))
#print(lj_params)
eps_pair, sigma_pair = pairwise_lj_params(symbols, lj_params,guest = (158.5, 3.72))
seed = 100
import numpy as np
rng = np.random.default_rng(seed)
boltzmann_mean_value,sem= widom(N, T, positions, eps_pair, sigma_pair, L, rng , n_blocks=10)
random_positions = rng.random((N,3))*L
energies = np.array([
    energy_fast(pos, positions, eps_pair, sigma_pair, L)
    for pos in random_positions
])
boltzmann_mean = np.mean(boltzmann(energies, T))

print(henry_coef(boltzmann_mean_value,T, rho_fw=rho_framework(cif_path),))

3.879675831730755e-06


In [5]:
positions, symbols, L, masses, atoms = load_framework(cif_path)
avogadro=6.02214076*10**(23)
total_mass = np.sum(masses)/1000/avogadro 
V_framework = (L*10**(-10))**3 
print(total_mass/V_framework)

593.3086673449801


In [4]:
seed = 100
import numpy as np
rng = np.random.default_rng(seed)
cif_path = "/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/data/IRMOF-1.cif"
positions, symbols, L, masses, atoms = load_framework(cif_path)
print(L)
#print(rng.random((100,3))*L)


25.832
